# Ensemble Method with Federated Training (Rotation + Dirichlet)

This notebook implements the ensemble clustering method with **TRUE federated learning** on CIFAR-10:
- **Feature heterogeneity**: Rotation-based transformations
- **Label heterogeneity**: Dirichlet distribution with tunable alpha parameter

**Key Difference from Standard Ensemble:**
- **Standard**: Pools data within clusters → centralized training
- **This version**: Trains each client separately → aggregates weights (FedAvg-style)

**Three phases:**
1. **Warmup**: All clients train local models with rotation + label imbalance
2. **Clustering**: Group clients using weight differences (FC + Layer4)
3. **Federated Hierarchical Training**: Train clients separately within clusters, then aggregate

**Why this matters:**
- Respects data privacy (no data pooling)
- Shows impact of label heterogeneity (alpha) on performance
- True federated learning within hierarchical structure

In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    from google.colab import drive
    drive.mount('/content/drive')
    
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

## ⚠️ Memory Warning for Colab Users

**If running on Google Colab Free:**
- Default settings may cause memory crashes (12-15GB RAM limit)
- **Recommended config changes:**
  - `batch_size = 32` (instead of 64)
  - `num_clients = 20` (keep reasonable)
  - `warmup_epochs = 1` (instead of 2-5)
  
Edit `config.json` before running if needed.

## Import Libraries

In [ ]:
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.metrics import confusion_matrix, silhouette_score, adjusted_rand_score
from sklearn.cluster import KMeans
import copy
import random
import time
from collections import defaultdict
import gc

sys.path.append('..')
from training.utils import get_model, set_seed
from training.ensemble_model import EnsembleModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Memory cleanup
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

## Load Configuration

In [ ]:
# Load configuration from JSON
import os
if os.path.exists('config.json'):
    config_path = 'config.json'
elif os.path.exists('experiments/config.json'):
    config_path = 'experiments/config.json'
else:
    raise FileNotFoundError("config.json not found")

with open(config_path, 'r') as f:
    CONFIG = json.load(f)

# Add Dirichlet alpha parameter
CONFIG['dirichlet_alpha'] = 0.5  # ← TUNE THIS: 0.1 (high non-IID) to 10.0 (low non-IID)

# Set random seeds
SEED = CONFIG['seed']
set_seed(SEED)
CONFIG['seed'] = SEED

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

# Memory monitoring
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB total")
    print(f"GPU Memory allocated: {torch.cuda.memory_allocated() / 1e9:.4f} GB")

## Create Rotation-Based Dataset

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """CIFAR-10 dataset with rotation applied."""
    def __init__(self, base_dataset, rotation_angle):
        self.base_dataset = base_dataset
        self.rotation_angle = rotation_angle
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        image = self.transform(image)
        return image, label

# Load CIFAR-10
print("Loading CIFAR-10 dataset...")
train_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)
test_dataset_raw = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=None)

print(f"Train dataset size: {len(train_dataset_raw)}")
print(f"Test dataset size: {len(test_dataset_raw)}")

## Distribute Data with Rotation + Dirichlet

Combine two types of heterogeneity:
1. **Rotation-based feature heterogeneity**: Assign clients to rotation clusters
2. **Dirichlet label heterogeneity**: Use Dirichlet distribution to create label imbalance

In [ ]:
# Generate rotation angles dynamically
rotation_angles = [int(360 * i / CONFIG['num_rotation_clusters']) for i in range(CONFIG['num_rotation_clusters'])]
print(f"Rotation angles: {rotation_angles}°")

# Assign clients to rotations
clients_per_rotation = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
client_rotation_labels = []

for rotation_idx, angle in enumerate(rotation_angles):
    start_client = rotation_idx * clients_per_rotation
    end_client = start_client + clients_per_rotation
    
    if rotation_idx == len(rotation_angles) - 1:
        end_client = CONFIG['num_clients']
    
    for client_idx in range(start_client, end_client):
        client_rotation_labels.append(angle)

print(f"\nClient distribution across rotations:")
for angle in rotation_angles:
    count = client_rotation_labels.count(angle)
    print(f"  {angle}°: {count} clients")

# Split ratios for train/val/test
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

print(f"\nData split ratios: Train={train_ratio}, Val={val_ratio}, Test={test_ratio}")

# Organize data by class
num_classes = 10
indices_by_class = [[] for _ in range(num_classes)]

for idx, (_, label) in enumerate(train_dataset_raw):
    indices_by_class[label].append(idx)

print(f"\nTotal samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: {len(indices_by_class[class_id])} samples")

# STEP 1: Split each class into train/val/test FIRST
train_indices_by_class = [[] for _ in range(num_classes)]
val_indices_by_class = [[] for _ in range(num_classes)]
test_indices_by_class = [[] for _ in range(num_classes)]

for class_id in range(num_classes):
    class_indices = np.array(indices_by_class[class_id])
    np.random.shuffle(class_indices)
    
    n = len(class_indices)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_indices_by_class[class_id] = class_indices[:train_end].tolist()
    val_indices_by_class[class_id] = class_indices[train_end:val_end].tolist()
    test_indices_by_class[class_id] = class_indices[val_end:].tolist()

print(f"\nSplit samples per class:")
for class_id in range(num_classes):
    print(f"  Class {class_id}: Train={len(train_indices_by_class[class_id])}, Val={len(val_indices_by_class[class_id])}, Test={len(test_indices_by_class[class_id])}")

# STEP 2: Apply Dirichlet distribution to train and val ONLY (test is IID)
print(f"\nApplying Dirichlet distribution (alpha={CONFIG['dirichlet_alpha']}) to train and val...")
print(f"Test set will be IID (uniform distribution)")

def distribute_with_dirichlet(indices_by_class, num_clients, alpha):
    """Distribute data to clients using Dirichlet distribution."""
    client_indices = [[] for _ in range(num_clients)]
    
    for class_id in range(len(indices_by_class)):
        class_indices = np.array(indices_by_class[class_id])
        np.random.shuffle(class_indices)
        
        # Sample proportions from Dirichlet distribution
        proportions = np.random.dirichlet(alpha=[alpha] * num_clients)
        proportions = (np.cumsum(proportions) * len(class_indices)).astype(int)[:-1]
        
        # Split indices according to proportions
        splits = np.split(class_indices, proportions)
        
        for client_idx, split in enumerate(splits):
            client_indices[client_idx].extend(split.tolist())
    
    # Shuffle each client's indices
    for client_idx in range(num_clients):
        random.shuffle(client_indices[client_idx])
    
    return client_indices

def distribute_iid(indices_by_class, num_clients):
    """Distribute data uniformly (IID) across clients."""
    # Flatten all indices
    all_indices = []
    for class_indices in indices_by_class:
        all_indices.extend(class_indices)
    
    # Shuffle
    random.shuffle(all_indices)
    
    # Split uniformly
    client_indices = [[] for _ in range(num_clients)]
    samples_per_client = len(all_indices) // num_clients
    
    for client_idx in range(num_clients):
        start = client_idx * samples_per_client
        end = start + samples_per_client if client_idx < num_clients - 1 else len(all_indices)
        client_indices[client_idx] = all_indices[start:end]
    
    return client_indices

# Apply Dirichlet to train and val, IID to test
client_indices_train = distribute_with_dirichlet(train_indices_by_class, CONFIG['num_clients'], CONFIG['dirichlet_alpha'])
client_indices_val = distribute_with_dirichlet(val_indices_by_class, CONFIG['num_clients'], CONFIG['dirichlet_alpha'])
client_indices_test = distribute_iid(test_indices_by_class, CONFIG['num_clients'])

# Create rotated datasets for each client
train_subsets = []
val_subsets = []
test_subsets = []
client_label_distributions = []

print(f"\nCreating datasets with rotations...")

for client_idx in range(CONFIG['num_clients']):
    angle = client_rotation_labels[client_idx]
    
    # Create rotated dataset (same rotation for train/val/test of this client)
    rotated_dataset = RotatedCIFAR10Dataset(train_dataset_raw, angle)
    
    # Create subsets
    train_subset = Subset(rotated_dataset, client_indices_train[client_idx])
    val_subset = Subset(rotated_dataset, client_indices_val[client_idx])
    test_subset = Subset(rotated_dataset, client_indices_test[client_idx])
    
    train_subsets.append(train_subset)
    val_subsets.append(val_subset)
    test_subsets.append(test_subset)
    
    # Track label distribution for this client (using train data)
    labels = [train_dataset_raw[idx][1] for idx in client_indices_train[client_idx]]
    label_dist = np.bincount(labels, minlength=num_classes)
    client_label_distributions.append(label_dist)

print(f"\nCreated {len(train_subsets)} client train datasets")
print(f"Created {len(val_subsets)} client validation datasets")
print(f"Created {len(test_subsets)} client test datasets")
print(f"\nAverage samples per client:")
print(f"  Train: {np.mean([len(s) for s in train_subsets]):.1f}")
print(f"  Validation: {np.mean([len(s) for s in val_subsets]):.1f}")
print(f"  Test: {np.mean([len(s) for s in test_subsets]):.1f}")
print(f"\nTotal samples:")
print(f"  Train: {sum([len(s) for s in train_subsets])}")
print(f"  Validation: {sum([len(s) for s in val_subsets])}")
print(f"  Test: {sum([len(s) for s in test_subsets])}")
print(f"  Total: {sum([len(s) for s in train_subsets]) + sum([len(s) for s in val_subsets]) + sum([len(s) for s in test_subsets])}")
print(f"\n✓ Train & Val: Dirichlet (non-IID, alpha={CONFIG['dirichlet_alpha']})")
print(f"✓ Test: IID (uniform distribution)")


## Visualize Data Distribution

In [ ]:
# Visualize label distribution heterogeneity
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Samples per client
ax = axes[0, 0]
client_sizes = [len(subset) for subset in train_subsets]
ax.bar(range(CONFIG['num_clients']), client_sizes, alpha=0.7)
ax.axhline(y=np.mean(client_sizes), color='red', linestyle='--', label=f'Mean: {np.mean(client_sizes):.0f}')
ax.set_xlabel('Client ID')
ax.set_ylabel('Number of Samples')
ax.set_title(f'Data Distribution Across Clients\n(Std: {np.std(client_sizes):.1f})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Label distribution heatmap
ax = axes[0, 1]
label_dist_matrix = np.array(client_label_distributions).T
im = ax.imshow(label_dist_matrix, cmap='YlOrRd', aspect='auto')
ax.set_xlabel('Client ID')
ax.set_ylabel('Class Label')
ax.set_title(f'Label Distribution per Client\n(Dirichlet α={CONFIG["dirichlet_alpha"]})')
plt.colorbar(im, ax=ax, label='Sample Count')

# Plot 3: Rotation distribution
ax = axes[1, 0]
rotation_counts = [client_rotation_labels.count(angle) for angle in rotation_angles]
ax.bar([f"{angle}°" for angle in rotation_angles], rotation_counts, alpha=0.7, color='steelblue')
ax.set_xlabel('Rotation Angle')
ax.set_ylabel('Number of Clients')
ax.set_title('Clients per Rotation Cluster')
ax.grid(True, alpha=0.3, axis='y')

# Plot 4: Label entropy per client (measure of balance)
ax = axes[1, 1]
entropies = []
for dist in client_label_distributions:
    probs = dist / dist.sum()
    probs = probs[probs > 0]  # Remove zeros
    entropy = -np.sum(probs * np.log2(probs))
    entropies.append(entropy)

ax.hist(entropies, bins=20, alpha=0.7, edgecolor='black')
ax.axvline(x=np.log2(num_classes), color='red', linestyle='--', label=f'Max (uniform): {np.log2(num_classes):.2f}')
ax.set_xlabel('Entropy (bits)')
ax.set_ylabel('Number of Clients')
ax.set_title(f'Label Distribution Entropy\n(Mean: {np.mean(entropies):.2f}, Lower = more imbalanced)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ensemble_federated_data_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nLabel distribution statistics:")
print(f"  Mean entropy: {np.mean(entropies):.3f} bits")
print(f"  Max possible entropy: {np.log2(num_classes):.3f} bits (uniform)")
print(f"  Entropy std: {np.std(entropies):.3f}")

## Create Validation and Test Datasets

In [ ]:
# Create validation dataset (combine all client validation sets WITH rotations)
print("Creating validation dataset...")
val_dataset = torch.utils.data.ConcatDataset(val_subsets)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Validation dataset size: {len(val_dataset)}")

# Create test dataset (combine all client test sets WITH rotations)
print("Creating test dataset...")
test_dataset = torch.utils.data.ConcatDataset(test_subsets)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Test dataset size: {len(test_dataset)}")
print(f"\n✓ All splits (train/val/test) have SAME rotation and label distributions")
print(f"⚠️  Test set will ONLY be evaluated at the end (proper ML practice)")

## Phase 1: Warmup Training

Each client trains a local model on their rotation + label distribution for a few epochs.

In [ ]:
def train_local_model(model, train_loader, epochs, lr):
    """Train local model for specified epochs."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model

print("Warmup training function defined")

In [ ]:
# Warmup configuration
warmup_epochs = CONFIG['warmup_epochs']

print(f"{'='*70}")
print(f"PHASE 1: WARMUP TRAINING (Rotation + Dirichlet α={CONFIG['dirichlet_alpha']})")
print(f"{'='*70}")
print(f"Number of clients: {CONFIG['num_clients']}")
print(f"Warmup epochs: {warmup_epochs}")
print(f"{'='*70}\n")

client_models = []
warmup_start = time.time()

for client_idx in range(CONFIG['num_clients']):
    # Create local model
    local_model = get_model(
        model_name=CONFIG['model_name'],
        num_classes=10,
        pretrained=CONFIG['pretrained']
    ).to(device)
    
    # Create data loader
    train_loader = DataLoader(
        train_subsets[client_idx],
        batch_size=CONFIG['batch_size'],
        shuffle=True
    )
    
    # Train locally
    local_model = train_local_model(local_model, train_loader, warmup_epochs, CONFIG['lr'])
    
    # Store model
    client_models.append(local_model)
    
    if (client_idx + 1) % 10 == 0:
        print(f"Completed warmup for {client_idx + 1}/{CONFIG['num_clients']} clients")

warmup_time = time.time() - warmup_start

print(f"\n{'='*70}")
print(f"Warmup Complete!")
print(f"{'='*70}")
print(f"Time: {warmup_time:.2f}s ({warmup_time/60:.2f} min)")
print(f"Avg time per client: {warmup_time/CONFIG['num_clients']:.2f}s")

# Memory cleanup after warmup
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

if torch.cuda.is_available():
    print(f"GPU Memory allocated after warmup: {torch.cuda.memory_allocated() / 1e9:.4f} GB")

## Phase 2: Extract Weights and Cluster Clients

Extract FC and Layer4 weights from all clients and cluster them using K-Means.

In [ ]:
def extract_fc_layer4_weights(model):
    """Extract FC layer and Layer4 weights as flattened vector."""
    weight_vector = []
    
    state_dict = model.state_dict()
    
    # Extract Layer4 weights (ResNet18)
    for key in state_dict.keys():
        if 'layer4' in key and ('weight' in key or 'bias' in key):
            weight_vector.append(state_dict[key].flatten().cpu().numpy())
    
    # Extract FC layer weights
    if 'fc.weight' in state_dict:
        weight_vector.append(state_dict['fc.weight'].flatten().cpu().numpy())
    if 'fc.bias' in state_dict:
        weight_vector.append(state_dict['fc.bias'].flatten().cpu().numpy())
    
    return np.concatenate(weight_vector)

print("Weight extraction function defined")

In [ ]:
print(f"{'='*70}")
print("PHASE 2: CLUSTERING")
print(f"{'='*70}")

# Extract weights from all client models
print("Extracting weights from all clients...")
client_weight_vectors = []

for client_idx, model in enumerate(client_models):
    weight_vec = extract_fc_layer4_weights(model)
    client_weight_vectors.append(weight_vec)

client_weight_matrix = np.array(client_weight_vectors)
print(f"Weight matrix shape: {client_weight_matrix.shape}")

# Compute weight differences from mean
mean_weights = client_weight_matrix.mean(axis=0)
weight_differences = client_weight_matrix - mean_weights

# Normalize
weight_differences = weight_differences / (np.linalg.norm(weight_differences, axis=1, keepdims=True) + 1e-8)

print(f"Normalized weight differences shape: {weight_differences.shape}")

# K-Means clustering
num_clusters = CONFIG['num_clusters_model']
print(f"\nApplying K-Means clustering (K={num_clusters})...")

kmeans = KMeans(n_clusters=num_clusters, random_state=SEED, n_init=10)
cluster_labels = kmeans.fit_predict(weight_differences)

# Cluster statistics
print(f"\nCluster assignment:")
for cluster_id in range(num_clusters):
    members = np.where(cluster_labels == cluster_id)[0]
    print(f"  Cluster {cluster_id}: {len(members)} clients")
    
    # Show rotation distribution in this cluster
    rotation_dist = {}
    for client_idx in members:
        angle = client_rotation_labels[client_idx]
        rotation_dist[angle] = rotation_dist.get(angle, 0) + 1
    print(f"    Rotation distribution: {rotation_dist}")

# Clustering quality metrics
silhouette_avg = silhouette_score(weight_differences, cluster_labels)
print(f"\nSilhouette score: {silhouette_avg:.4f}")

# Compare with ground truth rotation labels
rotation_to_id = {angle: idx for idx, angle in enumerate(sorted(set(client_rotation_labels)))}
ground_truth_labels = [rotation_to_id[angle] for angle in client_rotation_labels]

# Only compute ARI if num_clusters matches num ground truth clusters
if num_clusters == len(rotation_angles):
    ari_score = adjusted_rand_score(ground_truth_labels, cluster_labels)
    print(f"Adjusted Rand Index (vs rotations): {ari_score:.4f}")
else:
    print(f"ARI not computed (num_clusters={num_clusters} != num_rotations={len(rotation_angles)})")

print(f"{'='*70}\n")

# Memory cleanup
del client_weight_vectors, client_weight_matrix, weight_differences
gc.collect()

## Phase 3: Federated Hierarchical Training

**KEY DIFFERENCE: True Federated Learning**

Instead of pooling data within clusters:
1. Each client in a cluster trains locally on their own data
2. After local training, aggregate weights within the cluster (FedAvg-style)
3. Update the cluster's feature extractor with aggregated weights

This preserves data privacy and shows the true impact of label heterogeneity!

In [ ]:
# Create ensemble model
print(f"{'='*70}")
print("PHASE 3: FEDERATED HIERARCHICAL TRAINING")
print(f"{'='*70}")

# Import ResNetFeatureExtractor
from training.ensemble_model import ResNetFeatureExtractor

# Create feature extractors for each cluster from cluster representatives
feature_extractors = []

print("\nInitializing feature extractors from cluster representatives...")
for cluster_id in range(num_clusters):
    cluster_members = np.where(cluster_labels == cluster_id)[0]
    
    # Use first member as representative
    representative_idx = cluster_members[0]
    representative_model = client_models[representative_idx]
    
    # Create feature extractor from representative model (removes FC layer)
    feature_extractor = ResNetFeatureExtractor(representative_model)
    feature_extractors.append(feature_extractor)
    
    print(f"  Cluster {cluster_id}: Initialized from client {representative_idx} ({len(cluster_members)} members)")

# Create ensemble model with the feature extractors
ensemble_model = EnsembleModel(
    models=feature_extractors,
    feature_dim=512,  # ResNet18 output
    num_classes=10
).to(device)

print("\nEnsemble model created with cluster-specific feature extractors")

# Memory cleanup - delete client models as we don't need them anymore
del client_models
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
def federated_average_weights(models):
    """
    Aggregate model weights using FedAvg (simple averaging).
    
    Args:
        models: List of PyTorch models to aggregate
        
    Returns:
        Aggregated state_dict
    """
    # Get state dicts from all models
    state_dicts = [model.state_dict() for model in models]
    
    # Average all parameters
    avg_state_dict = {}
    for key in state_dicts[0].keys():
        # Stack tensors and compute mean
        avg_state_dict[key] = torch.stack([sd[key].float() for sd in state_dicts]).mean(dim=0)
    
    return avg_state_dict


def train_client_model_epochs(model, train_loader, epochs, lr):
    """
    Train a client model for specified epochs.
    Returns the trained model.
    """
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    return model


print("Federated aggregation functions defined")

In [ ]:
# Federated Hierarchical training configuration
training_rounds = CONFIG['training_rounds']
local_epochs = CONFIG['local_epochs']  # Epochs per client per round
criterion = nn.CrossEntropyLoss()

print(f"Federated Hierarchical training rounds: {training_rounds}")
print(f"Local epochs per client: {local_epochs}")
print(f"Learning rate: {CONFIG['lr']}")

# Calculate total training budget
warmup_budget = warmup_epochs * CONFIG['num_clients']
hierarchical_budget = training_rounds * CONFIG['num_clients'] * local_epochs
total_budget = warmup_budget + hierarchical_budget
print(f"\nFederated Ensemble Training Budget:")
print(f"  Warmup: {warmup_epochs} epochs × {CONFIG['num_clients']} clients = {warmup_budget} client-epochs")
print(f"  Hierarchical: {training_rounds} rounds × {CONFIG['num_clients']} clients × {local_epochs} epochs = {hierarchical_budget} client-epochs")
print(f"  Total: {total_budget} client-epochs")
print(f"{'='*70}\n")

# Storage for metrics
val_losses = []
val_accs = []
round_times = []

# Training loop
training_start = time.time()

for round_num in range(1, training_rounds + 1):
    round_start = time.time()
    
    # Train on all clusters (federated style)
    for cluster_id in range(num_clusters):
        cluster_members = np.where(cluster_labels == cluster_id)[0]
        
        # Get current cluster model weights
        cluster_feature_extractor = ensemble_model.models[cluster_id]
        
        # Create temporary client models for this cluster
        client_models_cluster = []
        
        # Each client trains separately on their own data
        for client_idx in cluster_members:
            # Create a copy of the cluster model for this client
            client_model = get_model(
                model_name=CONFIG['model_name'],
                num_classes=10,
                pretrained=False
            ).to(device)
            
            # Load cluster model state into client model's backbone
            # (We need to handle the ensemble structure carefully)
            cluster_backbone_state = cluster_feature_extractor.state_dict()
            client_state = client_model.state_dict()
            
            # Copy backbone weights from cluster model to client model
            for key in cluster_backbone_state.keys():
                if key in client_state:
                    client_state[key] = cluster_backbone_state[key]
            
            # Copy classifier weights from ensemble model
            client_state['fc.weight'] = ensemble_model.classifier.weight.data.clone()
            client_state['fc.bias'] = ensemble_model.classifier.bias.data.clone()
            
            client_model.load_state_dict(client_state)
            
            # Create data loader for this client
            client_loader = DataLoader(
                train_subsets[client_idx],
                batch_size=CONFIG['batch_size'],
                shuffle=True
            )
            
            # Train client model for local_epochs
            client_model = train_client_model_epochs(
                client_model, 
                client_loader, 
                local_epochs, 
                CONFIG['lr']
            )
            
            client_models_cluster.append(client_model)
        
        # Aggregate client models using FedAvg
        aggregated_state = federated_average_weights(client_models_cluster)
        
        # Update cluster feature extractor with aggregated weights (backbone only)
        cluster_state = cluster_feature_extractor.state_dict()
        for key in cluster_state.keys():
            if key in aggregated_state:
                cluster_state[key] = aggregated_state[key]
        cluster_feature_extractor.load_state_dict(cluster_state)
        
        # Update shared classifier with aggregated FC weights
        if 'fc.weight' in aggregated_state:
            ensemble_model.classifier.weight.data = aggregated_state['fc.weight']
        if 'fc.bias' in aggregated_state:
            ensemble_model.classifier.bias.data = aggregated_state['fc.bias']
        
        # Clean up client models to save memory
        del client_models_cluster
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Evaluate on VALIDATION set (not test!)
    ensemble_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Average predictions from all feature extractors
            batch_predictions = []
            for cluster_id in range(num_clusters):
                outputs = ensemble_model(inputs, [cluster_id])
                batch_predictions.append(outputs)
            
            # Average logits
            avg_outputs = torch.stack(batch_predictions).mean(dim=0)
            loss = criterion(avg_outputs, labels)
            
            # Accumulate loss weighted by batch size for proper averaging
            val_loss += loss.item() * inputs.size(0)
            _, predicted = avg_outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_val_loss = val_loss / total
    val_acc = correct / total
    
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)
    
    round_time = time.time() - round_start
    round_times.append(round_time)
    
    # Print progress
    if round_num % 5 == 0 or round_num == 1:
        print(f"Round {round_num}/{training_rounds} - "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}, "
              f"Time: {round_time:.2f}s")

training_time = time.time() - training_start
total_time = warmup_time + training_time

print(f"\n{'='*70}")
print(f"Federated Hierarchical Training Complete!")
print(f"{'='*70}")
print(f"Training time: {training_time:.2f}s ({training_time/60:.2f} min)")
print(f"Total time (warmup + training): {total_time:.2f}s ({total_time/60:.2f} min)")
print(f"Average time per round: {np.mean(round_times):.2f}s")
print(f"Final validation accuracy: {val_accs[-1]:.4f}")
print(f"Best validation accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")

## Final Test Set Evaluation

Now evaluate on the held-out test set (only evaluated ONCE at the end).

In [ ]:
# Final evaluation on TEST set (only once!)
print(f"{'='*70}")
print("FINAL TEST SET EVALUATION")
print(f"{'='*70}\n")

ensemble_model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Average predictions from all feature extractors
        batch_predictions = []
        for cluster_id in range(num_clusters):
            outputs = ensemble_model(inputs, [cluster_id])
            batch_predictions.append(outputs)
        
        # Average logits
        avg_outputs = torch.stack(batch_predictions).mean(dim=0)
        loss = criterion(avg_outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        _, predicted = avg_outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

final_test_loss = test_loss / total
final_test_acc = correct / total

print(f"Final Test Loss: {final_test_loss:.4f}")
print(f"Final Test Accuracy: {final_test_acc:.4f}")
print(f"\nComparison:")
print(f"  Best Validation Accuracy: {max(val_accs):.4f} (round {np.argmax(val_accs)+1})")
print(f"  Final Test Accuracy: {final_test_acc:.4f}")
print(f"  Difference (Val - Test): {max(val_accs) - final_test_acc:+.4f}")
print(f"{'='*70}\n")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score
from sklearn.preprocessing import label_binarize
import itertools

# Collect predictions and labels on test set
print("Collecting predictions on test set...")
ensemble_model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Average predictions from all feature extractors
        batch_predictions = []
        for cluster_id in range(num_clusters):
            outputs = ensemble_model(inputs, [cluster_id])
            batch_predictions.append(outputs)
        
        # Average logits
        avg_outputs = torch.stack(batch_predictions).mean(dim=0)
        
        # Get probabilities for AUC
        probs = torch.nn.functional.softmax(avg_outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
        
        _, predicted = avg_outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_probs = np.concatenate(all_probs, axis=0)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"\n{'='*70}")
print("DETAILED TEST SET METRICS")
print(f"{'='*70}\n")

# 1. Classification Report (Precision, Recall, F1-Score per class)
print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

# 2. Overall Metrics
macro_f1 = f1_score(all_labels, all_preds, average='macro')
micro_f1 = f1_score(all_labels, all_preds, average='micro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"\nF1-Score Summary:")
print(f"  Macro F1:    {macro_f1:.4f} (unweighted mean)")
print(f"  Micro F1:    {micro_f1:.4f} (overall accuracy)")
print(f"  Weighted F1: {weighted_f1:.4f} (weighted by support)")

# 3. ROC-AUC Score (One-vs-Rest)
try:
    # Binarize labels for multi-class AUC
    labels_binarized = label_binarize(all_labels, classes=range(10))
    
    # Macro-average AUC (average of per-class AUC)
    macro_auc = roc_auc_score(labels_binarized, all_probs, average='macro')
    
    # Weighted-average AUC
    weighted_auc = roc_auc_score(labels_binarized, all_probs, average='weighted')
    
    # Per-class AUC
    per_class_auc = roc_auc_score(labels_binarized, all_probs, average=None)
    
    print(f"\nROC-AUC Summary:")
    print(f"  Macro AUC:    {macro_auc:.4f}")
    print(f"  Weighted AUC: {weighted_auc:.4f}")
    print(f"\nPer-class AUC:")
    for i, name in enumerate(class_names):
        print(f"  {name:12s}: {per_class_auc[i]:.4f}")
        
except Exception as e:
    print(f"\nNote: Could not compute AUC scores: {e}")

print(f"\n{'='*70}\n")

## Confusion Matrix Visualization

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(all_labels, all_preds)

# Calculate percentage confusion matrix
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Plot 1: Confusion Matrix (Counts)
ax = axes[0]
im1 = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im1, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Counts\nFederated Ensemble (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm[i, j], 'd'),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
            fontsize=9)

# Plot 2: Confusion Matrix (Percentages)
ax = axes[1]
im2 = ax.imshow(cm_percent, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im2, ax=ax, format='%.1f%%')
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=class_names,
       yticklabels=class_names,
       xlabel='Predicted Label',
       ylabel='True Label',
       title=f'Confusion Matrix - Percentages\nFederated Ensemble (α={CONFIG["dirichlet_alpha"]})')
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Add text annotations
thresh = cm_percent.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, format(cm_percent[i, j], '.1f'),
            ha="center", va="center",
            color="white" if cm_percent[i, j] > thresh else "black",
            fontsize=9)

plt.tight_layout()
plt.savefig(f'ensemble_federated_alpha{CONFIG["dirichlet_alpha"]}_confusion_matrix.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved")

# Calculate per-class accuracy from confusion matrix
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print(f"\nPer-class Accuracy:")
for i, name in enumerate(class_names):
    print(f"  {name:12s}: {per_class_accuracy[i]:.4f} ({per_class_accuracy[i]*100:.2f}%)")

## Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Validation loss curve
ax = axes[0]
rounds_range = range(1, training_rounds + 1)
ax.plot(rounds_range, val_losses, 'o-', linewidth=2, markersize=4)
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Validation Loss', fontsize=12)
ax.set_title(f'Federated Ensemble Validation Loss (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

# Plot 2: Validation accuracy curve
ax = axes[1]
ax.plot(rounds_range, val_accs, 's-', linewidth=2, markersize=4, color='green', label='Validation')
ax.axhline(y=max(val_accs), color='red', linestyle='--', alpha=0.5, 
           label=f'Best Val: {max(val_accs):.4f}')
ax.axhline(y=final_test_acc, color='blue', linestyle='--', alpha=0.5,
           label=f'Final Test: {final_test_acc:.4f}')
ax.set_xlabel('Round', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title(f'Federated Ensemble Validation Accuracy (α={CONFIG["dirichlet_alpha"]})', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(f'ensemble_federated_alpha{CONFIG["dirichlet_alpha"]}_curves.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training curves saved")

## Save Results

In [ ]:
# Save results to file
results = {
    'method': 'ensemble_federated_rotation_dirichlet',
    'config': CONFIG,
    'warmup_epochs': warmup_epochs,
    'training_rounds': training_rounds,
    'num_clusters': num_clusters,
    'dirichlet_alpha': CONFIG['dirichlet_alpha'],
    'training_type': 'federated',  # NEW: Mark as federated
    'times': {
        'warmup_time': warmup_time,
        'training_time': training_time,
        'total_time': total_time,
        'avg_round_time': float(np.mean(round_times))
    },
    'performance': {
        'best_val_acc': float(max(val_accs)),
        'best_val_round': int(np.argmax(val_accs) + 1),
        'final_val_acc': float(val_accs[-1]),
        'final_test_acc': float(final_test_acc),
        'final_test_loss': float(final_test_loss),
        'val_test_gap': float(max(val_accs) - final_test_acc)
    },
    'val_losses': [float(x) for x in val_losses],
    'val_accs': [float(x) for x in val_accs],
    'test_metrics': {
        'macro_f1': float(macro_f1),
        'micro_f1': float(micro_f1),
        'weighted_f1': float(weighted_f1),
        'macro_auc': float(macro_auc) if 'macro_auc' in locals() else None,
        'weighted_auc': float(weighted_auc) if 'weighted_auc' in locals() else None,
        'per_class_auc': [float(x) for x in per_class_auc] if 'per_class_auc' in locals() else None,
        'per_class_accuracy': [float(x) for x in per_class_accuracy],
        'confusion_matrix': cm.tolist()
    },
    'clustering': {
        'silhouette_score': float(silhouette_avg),
        'cluster_sizes': [int(np.sum(cluster_labels == i)) for i in range(num_clusters)]
    },
    'data_stats': {
        'mean_train_samples_per_client': float(np.mean([len(s) for s in train_subsets])),
        'mean_val_samples_per_client': float(np.mean([len(s) for s in val_subsets])),
        'total_train_samples': sum([len(s) for s in train_subsets]),
        'total_val_samples': sum([len(s) for s in val_subsets]),
        'mean_label_entropy': float(np.mean(entropies)),
        'std_label_entropy': float(np.std(entropies))
    }
}

# Add ARI if computed
if num_clusters == len(rotation_angles):
    results['clustering']['ari_score'] = float(ari_score)

filename = f'ensemble_federated_alpha{CONFIG["dirichlet_alpha"]}_results.json'
with open(filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to '{filename}'")

# Save model checkpoint
checkpoint_name = f'ensemble_federated_alpha{CONFIG["dirichlet_alpha"]}_checkpoint.pth'
torch.save({
    'training_rounds': training_rounds,
    'model_state_dict': ensemble_model.state_dict(),
    'cluster_labels': cluster_labels,
    'best_val_acc': max(val_accs),
    'final_test_acc': final_test_acc,
    'config': CONFIG
}, checkpoint_name)

print(f"Model checkpoint saved to '{checkpoint_name}'")

## Summary

**Federated Ensemble Method with Combined Heterogeneity:**

**Key Innovation:**
- **True Federated Learning**: Clients train separately, weights aggregated (no data pooling)
- **Privacy Preserving**: Raw data never leaves client devices
- **Hierarchical Structure**: Cluster-specific models + shared classifier

**Training Process:**
1. **Warmup**: Each client trains local model independently
2. **Clustering**: Group clients by weight similarity
3. **Federated Hierarchical**: 
   - Each cluster round:
     - All clients in cluster train locally on their own data
     - Aggregate client weights using FedAvg
     - Update cluster model with aggregated weights

**Expected Alpha Impact (Unlike Standard Ensemble):**
- **α = 0.1**: Extreme label imbalance → **performance degradation expected**
- **α = 0.5**: Moderate imbalance → moderate performance
- **α = 10.0**: Nearly balanced → best performance

**Comparison:**
- **Standard Ensemble**: Data pooling masks heterogeneity effects
- **Federated Ensemble**: True impact of label heterogeneity visible
- **FedAvg**: Global model struggles with heterogeneity
- **This Method**: Hierarchical + federated = best of both worlds